# Module D.4–D.6: RAG Generation — Reranking, Context & Capstone
**Part II — Applied LLM Engineering**

> Retrieve, rerank, construct context, generate. Your first end-to-end RAG system.

## 1. Where We Left Off

In NB34 (Module D.1–D.3) you built the **retrieval half** of RAG:

- **Dense retrieval**: embed query and corpus with a bi-encoder (`all-MiniLM-L6-v2`), rank by cosine similarity.
- **BM25**: sparse keyword matching — fast, no model, great for exact terms.
- **Hybrid search**: combine both scores for robustness.

The output of NB34 was a ranked list of candidate chunks. That's the *first stage*.

This notebook builds the **second stage — everything from candidates to answer**:

```
Query
  │
  ▼
[Stage 1 — Retrieval]   bi-encoder + BM25 → top-K candidates   (fast, recall-focused)
  │
  ▼
[Stage 2 — Reranking]   cross-encoder → top-k' best chunks      (slow, precision-focused)
  │
  ▼
[Stage 3 — Context]     build context string within token budget
  │
  ▼
[Stage 4 — Generation]  LLM reads context, generates grounded answer
```

By the end you'll have a working `RAGSystem` class you can reuse on any corpus.

## 2. Setup — Corpus, Models, and Helper

Run this cell first. It loads all three models and defines the `chat()` helper used throughout.

**Three models, three roles:**

| Model | Role | Speed |
|---|---|---|
| `all-MiniLM-L6-v2` | Bi-encoder for dense retrieval | Very fast (batch embed) |
| `cross-encoder/ms-marco-MiniLM-L-6-v2` | Cross-encoder for reranking | ~10× slower per pair |
| `HuggingFaceTB/SmolLM2-135M-Instruct` | Generator LLM | Autoregressive (slow per token) |

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import torch
import re

# ---------------------------------------------------------------------------
# Arborian corpus — a fictional civilisation that lives in ancient trees.
# These 15 chunks are the knowledge base our RAG system will answer from.
# Each chunk is a short, self-contained fact.
# ---------------------------------------------------------------------------
CORPUS = [
    "The Arborian currency is called the Leaflet. One Leaflet is subdivided into 100 Sprigs.",
    "Arborians build their homes inside and on top of ancient redwood trees, hollowing out chambers while preserving the trees' health.",
    "The capital city of Arboria is Canopy, situated at the crown of the Great Redwood.",
    "The Bark Elder is the supreme leader of the Arborian people, chosen by a council of eldest trees — or rather, by those who speak for them.",
    "Every spring, Arborians celebrate the Festival of Leaves, a week-long event of music, storytelling, and offerings of carved wood.",
    "The Arborian written language uses stylised leaf shapes. Each species of leaf represents a different phoneme.",
    "Arboria's primary export is hand-carved wood sculptures, traded with neighbouring civilisations for metal tools and cloth.",
    "The Arborian diet consists mainly of bark bread, forest mushrooms, sap wine, and fruit harvested from the canopy.",
    "Arborian medicine relies on poultices made from tree resin and moss. The Head Healer is called the Root Doctor.",
    "Children in Arboria learn to climb before they learn to walk on flat ground. Schools are suspended platforms between branches.",
    "The Festival of Leaves includes a ritual where every family burns a single carved Leaflet coin to honour the forest spirits.",
    "The Great Redwood, home to Canopy, is estimated to be over 3,000 years old and is considered sacred.",
    "Arborian law forbids felling any living tree. Timber may only be collected from trees that have fallen naturally.",
    "Travel between Arborian settlements is conducted along rope bridges strung between the tallest trees.",
    "The Bark Elder's annual address is delivered at the Festival of Leaves from the highest branch of the Great Redwood.",
]

print(f"Corpus: {len(CORPUS)} chunks loaded.")
print("\nFirst chunk:", CORPUS[0])

# ---------------------------------------------------------------------------
# Model 1: Bi-encoder for dense retrieval
# ---------------------------------------------------------------------------
print("\nLoading bi-encoder...")
biencoder = SentenceTransformer("all-MiniLM-L6-v2")
# Pre-compute corpus embeddings once
corpus_embeddings = biencoder.encode(CORPUS, convert_to_numpy=True, normalize_embeddings=True)
print(f"Corpus embeddings shape: {corpus_embeddings.shape}")

# ---------------------------------------------------------------------------
# Model 2: Cross-encoder for reranking (loaded in Section 3)
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Model 3: Generator LLM
# ---------------------------------------------------------------------------
print("\nLoading generator LLM...")
MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
llm.eval()
print(f"Generator loaded. Parameters: {sum(p.numel() for p in llm.parameters()) / 1e6:.0f} M")


def chat(messages, max_new_tokens=200, temperature=0.0):
    """Send messages to SmolLM2-Instruct and return the assistant reply."""
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    )
    do_sample = bool(temperature and temperature > 0)
    out = llm.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=(temperature if do_sample else None),
        pad_token_id=tok.eos_token_id
    )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print("\nAll models ready.")

## 3. Why First-Stage Retrieval Isn't Enough

The bi-encoder from NB34 encodes the query and each document **independently**, then computes cosine similarity. This makes it fast — you embed once per document at index time and once per query at search time — but it introduces a fundamental precision problem.

**The problem**: the query and document are never compared *together*. The model compresses them each into a single 384-dimensional vector without seeing the other. Nuances get lost.

**The cross-encoder fix**: a cross-encoder receives `(query, document)` as a *single input*, so attention layers can directly model the interaction between every query token and every document token. This is far more precise — but requires a forward pass per `(query, doc)` pair, making it O(n) instead of O(1) in retrieval.

```
Bi-encoder:   encode(query) • encode(doc)   — independent, fast
Cross-encoder: score( [query] [SEP] [doc] ) — joint, slow but precise
```

**Classic two-stage pipeline**:
1. **Retrieve**: bi-encoder fetches top-20 candidates. Optimise for **recall** — don't miss the answer.
2. **Rerank**: cross-encoder scores all 20 pairs, keeps top-5. Optimise for **precision** — put the best chunk first.

You run the expensive cross-encoder on only 20 docs, not the entire corpus.

## 4. Cross-Encoder Reranking

Load `cross-encoder/ms-marco-MiniLM-L-6-v2`, a small BERT-based model fine-tuned on the MS MARCO passage ranking dataset. It scores relevance on a continuous scale — higher = more relevant.

Its interface is simple: pass a list of `(query, document)` tuples and get back one float score per pair.

In [ ]:
print("Loading cross-encoder...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder ready.")

# Interface demonstration
demo_query = "What is the Arborian currency?"
demo_docs = [
    "The Arborian currency is called the Leaflet.",
    "Arborians build homes in ancient redwood trees.",
    "The capital city of Arboria is Canopy.",
]

pairs = [(demo_query, doc) for doc in demo_docs]
scores = reranker.predict(pairs)

print(f"\nQuery: {demo_query!r}")
print("\nCross-encoder scores (higher = more relevant):")
for doc, score in zip(demo_docs, scores):
    print(f"  {score:7.3f}  |  {doc}")

In [ ]:
# ---------------------------------------------------------------------------
# Helper: dense retrieval with bi-encoder
# ---------------------------------------------------------------------------
def dense_retrieve(query: str, k: int = 8) -> list[tuple[int, float]]:
    """Return list of (corpus_idx, cosine_score) sorted by score desc."""
    q_emb = biencoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = (corpus_embeddings @ q_emb.T).squeeze()  # cosine sim (normalised)
    top_indices = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_indices]


# ---------------------------------------------------------------------------
# Helper: rerank a set of (idx, score) candidates with the cross-encoder
# ---------------------------------------------------------------------------
def rerank(query: str, candidates: list[tuple[int, float]], top_k: int = 3) -> list[tuple[int, float]]:
    """Rerank candidates with the cross-encoder. Return top_k (idx, ce_score) pairs."""
    if not candidates:
        return []
    pairs = [(query, CORPUS[idx]) for idx, _ in candidates]
    ce_scores = reranker.predict(pairs)
    ranked = sorted(
        [(candidates[i][0], float(ce_scores[i])) for i in range(len(candidates))],
        key=lambda x: x[1], reverse=True
    )
    return ranked[:top_k]


# ---------------------------------------------------------------------------
# Demo: retrieve 8, then rerank to top-3
# ---------------------------------------------------------------------------
query = "What is the Arborian currency?"

stage1 = dense_retrieve(query, k=8)
stage2 = rerank(query, stage1, top_k=3)

print(f"Query: {query!r}")
print(f"\n--- Stage 1: Bi-encoder top-8 (cosine similarity) ---")
for rank, (idx, score) in enumerate(stage1, 1):
    marker = "  <-- top after rerank" if idx == stage2[0][0] else ""
    print(f"  [{rank}] score={score:.4f}  {CORPUS[idx][:70]}...{marker}" if len(CORPUS[idx]) > 70
          else f"  [{rank}] score={score:.4f}  {CORPUS[idx]}{marker}")

print(f"\n--- Stage 2: Cross-encoder top-3 (reranked) ---")
for rank, (idx, score) in enumerate(stage2, 1):
    print(f"  [{rank}] ce_score={score:.3f}  {CORPUS[idx][:70]}..." if len(CORPUS[idx]) > 70
          else f"  [{rank}] ce_score={score:.3f}  {CORPUS[idx]}")

**Why reranking helps here**: the bi-encoder may rank "The Festival of Leaves includes a ritual where every family burns a single carved *Leaflet* coin..." highly because the word "Leaflet" appears. But the cross-encoder correctly demotes it — the sentence is about the festival ritual, not about the currency system. The joint attention over `(query, doc)` tokens catches this distinction.

## 5. Context Window Budgeting

The generator has a finite context window. Even if you retrieved 20 excellent chunks, you can't send them all — the prompt would exceed the model's limit or crowd out the answer space.

**Token budgeting**: pack as many chunks as fit, stop when the budget runs out.

We use a fast approximation: `n_tokens ≈ n_words × 1.3`. English averages ~1.3 subword tokens per whitespace-separated word. This is a rough estimate — in production use the actual tokenizer.

In [ ]:
def estimate_tokens(text: str) -> int:
    """Fast word-count approximation: words * 1.3 rounds to int."""
    return int(len(text.split()) * 1.3)


def build_context(chunks: list[str], max_tokens: int = 400) -> str:
    """
    Greedily pack chunks into a context string that fits within max_tokens.
    Chunks are assumed to be already ranked (best first).
    Returns the assembled context string and the number of chunks included.
    """
    included = []
    used_tokens = 0
    for chunk in chunks:
        chunk_tokens = estimate_tokens(chunk)
        if used_tokens + chunk_tokens > max_tokens:
            break  # budget exhausted
        included.append(chunk)
        used_tokens += chunk_tokens
    return "\n\n".join(included), len(included), used_tokens


# Demonstrate on the full corpus (no retrieval, just budgeting)
print("Token budget experiment — how many corpus chunks fit in 400 tokens?")
print()
context_str, n_included, n_tokens = build_context(CORPUS, max_tokens=400)
print(f"Chunks included: {n_included} / {len(CORPUS)}")
print(f"Estimated tokens used: {n_tokens}")
print()

# Compare with actual tokenizer count
actual_tokens = len(tok.encode(context_str))
print(f"Actual tokenizer count: {actual_tokens}  (approximation error: {abs(actual_tokens - n_tokens)} tokens)")
print()

# Show how budget scales
print("Budget vs. chunks included:")
for budget in [100, 200, 400, 600]:
    _, n, used = build_context(CORPUS, max_tokens=budget)
    print(f"  budget={budget:4d} tokens → {n:2d} chunks ({used} tokens used)")

## 6. Context Construction — The RAG Prompt

How you frame the context matters as much as which chunks you include. A well-structured prompt:

1. **Delimits the context clearly** — so the model knows where external facts start and end.
2. **Instructs grounding** — "use only the context above" prevents the model mixing in its parametric knowledge.
3. **Provides a graceful fallback** — "If the context does not contain the answer, say I don't know" is critical for faithfulness. Without it, the model will hallucinate rather than admit ignorance.

The three-part structure below is a battle-tested pattern:

In [ ]:
def build_rag_prompt(query: str, chunks: list[str]) -> str:
    """
    Assemble a RAG prompt from ranked chunks and a question.

    Structure:
      [CONTEXT] ... [/CONTEXT]
      Grounding instruction.
      Question: ...
    """
    context_str, _, _ = build_context(chunks, max_tokens=400)
    prompt = (
        "[CONTEXT]\n"
        + context_str
        + "\n[/CONTEXT]\n\n"
        "Answer the following question using only the context above. "
        "If the context does not contain the answer, say \"I don't know.\""
        "\n\nQuestion: " + query
    )
    return prompt


# Show what the prompt looks like
demo_chunks = [
    CORPUS[0],  # Leaflet currency
    CORPUS[2],  # Capital Canopy
    CORPUS[6],  # Wood exports
]
demo_query = "What is the Arborian currency?"
print(build_rag_prompt(demo_query, demo_chunks))

print("\n" + "=" * 60)
print("Prompt token count:", len(tok.encode(build_rag_prompt(demo_query, demo_chunks))))

**Why these design choices?**

- `[CONTEXT]...[/CONTEXT]` — XML-style delimiters are visually unambiguous tokens that the model can unambiguously identify as boundaries. The model was trained on text containing these patterns and attends to them strongly.
- "use only the context above" — without this, the model will blend retrieved facts with parametric knowledge, making hallucination detection impossible.
- "If the context does not contain the answer, say I don't know" — this is the safety valve. A RAG system without a graceful fallback is worse than no RAG system, because it confidently generates plausible-sounding wrong answers.

## 7. Lost in the Middle

Research (Liu et al., 2023 — *Lost in the Middle: How Language Models Use Long Contexts*) shows that LLMs are biased toward the **beginning and end** of their context window. Information buried in the middle gets less attention weight in practice.

**Implication for RAG**: if you put your best chunk in the middle of 10 chunks, the model may ignore it.

**Mitigation**: after reranking, place rank-1 (most relevant) at the **top** of the context and rank-2 at the **bottom**. Middle positions get the least useful chunks.

In [ ]:
def order_chunks_for_context(ranked_chunks: list[str]) -> list[str]:
    """
    Reorder ranked chunks to mitigate the lost-in-the-middle effect.

    Strategy (Liu et al., 2023):
      - rank-1 (most relevant) → position 0 (first / top)
      - rank-2 (second most relevant) → last position (bottom)
      - remaining chunks fill the middle in descending order

    Args:
        ranked_chunks: chunks already sorted best-first by the reranker.

    Returns:
        Reordered list with best chunk first and second-best chunk last.
    """
    if len(ranked_chunks) <= 2:
        return ranked_chunks  # nothing to reorder

    best = ranked_chunks[0]
    second_best = ranked_chunks[1]
    middle = ranked_chunks[2:]  # ranked 3..n, placed in middle
    return [best] + middle + [second_best]


# Illustrate with 5 hypothetical chunks
sample = [f"Chunk {i+1} (rank {i+1})" for i in range(5)]
reordered = order_chunks_for_context(sample)

print("Original order (reranker output, best-first):")
for i, c in enumerate(sample):
    print(f"  position {i}: {c}")

print("\nReordered (lost-in-the-middle mitigation):")
for i, c in enumerate(reordered):
    label = "  ← most relevant" if i == 0 else ("  ← second-most relevant" if i == len(reordered) - 1 else "")
    print(f"  position {i}: {c}{label}")

## 8. Citations

Knowing the answer is good; knowing *where it came from* is better. Citations let users verify the model's output and help you debug retrieval failures.

**Implementation strategy**:
1. Number each chunk `[1]`, `[2]`, ... in the context.
2. Instruct the model to cite sources like `[1]` when it uses a fact.
3. Parse citation numbers from the output with a regex.

In [ ]:
def build_rag_prompt_with_citations(query: str, chunks: list[str]) -> str:
    """
    Build a RAG prompt where each chunk is numbered for citation.
    Instructs the model to cite chunk numbers inline as [1], [2], ...
    """
    # Number each chunk
    numbered = [f"[{i+1}] {chunk}" for i, chunk in enumerate(chunks)]
    context_str = "\n\n".join(numbered)

    prompt = (
        "[CONTEXT]\n"
        + context_str
        + "\n[/CONTEXT]\n\n"
        "Answer the following question using only the context above. "
        "Cite the source chunk numbers inline as [1], [2], etc. "
        "If the context does not contain the answer, say \"I don't know.\""
        "\n\nQuestion: " + query
    )
    return prompt


def parse_citations(answer: str) -> list[int]:
    """Extract all citation numbers from an answer string. Returns sorted list of ints."""
    matches = re.findall(r"\[(\d+)\]", answer)
    return sorted(set(int(m) for m in matches))


# Demo
currency_chunks = [
    CORPUS[0],   # [1] Leaflet currency
    CORPUS[10],  # [2] Festival ritual burning a Leaflet coin
    CORPUS[6],   # [3] Wood exports
]
query = "What is the Arborian currency?"
cited_prompt = build_rag_prompt_with_citations(query, currency_chunks)

print("Prompt sent to model:")
print("-" * 60)
print(cited_prompt)
print("-" * 60)

answer = chat(
    [{"role": "user", "content": cited_prompt}],
    max_new_tokens=120,
    temperature=0.0
)
print("\nModel answer:")
print(answer)

cited = parse_citations(answer)
print(f"\nCitations parsed: {cited}")
for c in cited:
    if 1 <= c <= len(currency_chunks):
        print(f"  [{c}] → {currency_chunks[c-1]}")

## 9. Capstone: End-to-End RAG System

Now assemble all the pieces into a single `RAGSystem` class. It wraps the full pipeline:
retrieve → rerank → build context → generate → return structured result.

For retrieval we implement a simple hybrid: combine dense scores with a BM25-style TF-IDF approximation using token overlap. This avoids adding a BM25 library dependency while still blending lexical and semantic signals.

In [ ]:
class RAGSystem:
    """
    End-to-end Retrieval-Augmented Generation system.

    Pipeline:
      1. Hybrid retrieval: dense (bi-encoder cosine) + lexical (token overlap)
      2. Reranking: cross-encoder re-scores top candidates
      3. Context construction: budget-aware packing with lost-in-the-middle mitigation
      4. Generation: LLM reads context, produces grounded answer
    """

    def __init__(
        self,
        corpus: list[str],
        retrieval_k: int = 10,
        rerank_k: int = 3,
        context_budget: int = 400,
        max_new_tokens: int = 150,
    ):
        self.corpus = corpus
        self.retrieval_k = retrieval_k
        self.rerank_k = rerank_k
        self.context_budget = context_budget
        self.max_new_tokens = max_new_tokens

        # Pre-compute dense embeddings
        self._embs = biencoder.encode(
            corpus, convert_to_numpy=True, normalize_embeddings=True
        )
        # Pre-tokenise corpus for lexical overlap
        self._corpus_tokens = [set(doc.lower().split()) for doc in corpus]

    # ------------------------------------------------------------------
    # Stage 1: Hybrid retrieval
    # ------------------------------------------------------------------
    def _hybrid_retrieve(self, query: str) -> list[tuple[int, float]]:
        """Combine dense cosine similarity with lexical token overlap."""
        q_emb = biencoder.encode(
            [query], convert_to_numpy=True, normalize_embeddings=True
        )
        dense_scores = (self._embs @ q_emb.T).squeeze()

        # Lexical: Jaccard overlap between query tokens and doc tokens
        q_tokens = set(query.lower().split())
        lex_scores = np.array([
            len(q_tokens & dt) / max(len(q_tokens | dt), 1)
            for dt in self._corpus_tokens
        ])

        # Normalise each to [0, 1] then blend 70% dense / 30% lexical
        d_norm = dense_scores / (dense_scores.max() + 1e-9)
        l_norm = lex_scores / (lex_scores.max() + 1e-9)
        hybrid = 0.7 * d_norm + 0.3 * l_norm

        top_idx = np.argsort(hybrid)[::-1][: self.retrieval_k]
        return [(int(i), float(hybrid[i])) for i in top_idx]

    # ------------------------------------------------------------------
    # Stage 2: Cross-encoder reranking
    # ------------------------------------------------------------------
    def _rerank(self, query: str, candidates: list[tuple[int, float]]) -> list[tuple[int, float]]:
        pairs = [(query, self.corpus[idx]) for idx, _ in candidates]
        ce_scores = reranker.predict(pairs)
        ranked = sorted(
            [(candidates[i][0], float(ce_scores[i])) for i in range(len(candidates))],
            key=lambda x: x[1],
            reverse=True,
        )
        return ranked[: self.rerank_k]

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------
    def answer(self, question: str) -> dict:
        """
        Run the full RAG pipeline for a question.

        Returns:
            dict with keys: question, answer, sources, top_reranker_score
        """
        # 1. Retrieve
        candidates = self._hybrid_retrieve(question)

        # 2. Rerank
        reranked = self._rerank(question, candidates)
        top_score = reranked[0][1] if reranked else 0.0

        # 3. Build context — extract chunks, apply lost-in-the-middle ordering
        ranked_chunks = [self.corpus[idx] for idx, _ in reranked]
        ordered_chunks = order_chunks_for_context(ranked_chunks)
        context_str, n_chunks, _ = build_context(ordered_chunks, self.context_budget)
        used_chunks = ordered_chunks[:n_chunks]

        # 4. Generate
        prompt = (
            "[CONTEXT]\n"
            + context_str
            + "\n[/CONTEXT]\n\n"
            "Answer the following question using only the context above. "
            "If the context does not contain the answer, say \"I don't know.\""
            "\n\nQuestion: " + question
        )
        answer_text = chat(
            [{"role": "user", "content": prompt}],
            max_new_tokens=self.max_new_tokens,
            temperature=0.0,
        )

        return {
            "question": question,
            "answer": answer_text.strip(),
            "sources": used_chunks,
            "top_reranker_score": round(top_score, 3),
        }


# Instantiate
print("Building RAGSystem...")
rag = RAGSystem(CORPUS, retrieval_k=10, rerank_k=3)
print("RAGSystem ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Capstone demo: 5 questions — 4 answerable, 1 not in corpus
# ---------------------------------------------------------------------------
questions = [
    "What is the Arborian currency?",          # Answer: Leaflet
    "Where do Arborians live?",                # Answer: ancient redwood trees
    "Who leads the Arborians?",                # Answer: Bark Elder
    "What is the Arborian festival called?",   # Answer: Festival of Leaves
    "What is the GDP of Arboria?",             # Answer: I don't know — not in corpus
]

results = []
for q in questions:
    result = rag.answer(q)
    results.append(result)

    print("=" * 70)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"   Reranker top score: {result['top_reranker_score']}")
    print(f"   Sources used ({len(result['sources'])}):")
    for i, src in enumerate(result['sources'], 1):
        print(f"     [{i}] {src}")
    print()

**Reading the results**:

- Questions 1–4 should retrieve the correct chunks and the model should produce grounded, factually correct answers. The reranker score will be noticeably higher for these.
- Question 5 (GDP) should retrieve chunks with low reranker scores. Because the answer is truly absent, a well-grounded model says "I don't know" — exactly the behaviour the prompt engineering is designed to elicit.
- If the model hallucinates on Q5, that's a generation failure — we'll examine it in the next section.

## 10. Failure Modes

RAG can fail at any stage. Understanding *where* it breaks tells you how to fix it.

**Three canonical failure modes**:

| Failure | Symptom | Root cause | Fix |
|---|---|---|---|
| Retrieval failure | Wrong chunks retrieved; correct chunk ranked low | Bad query formulation, vocabulary mismatch | Query expansion, better embeddings |
| Generation failure | Model ignores context and hallucinates | Weak instruction-following, context too long, bad prompt | Stronger model, better prompt, fewer chunks |
| Context truncation | Answer was in the corpus but cut off by budget | context_budget too small | Increase budget, use smaller chunks |

In [ ]:
# --- Failure Mode 1: Retrieval failure ---
# A vaguely worded query that doesn't match the corpus vocabulary well.
# "monetary denomination" is a valid synonym for currency but won't match
# "Leaflet" or "currency" in the corpus — dense retrieval may mis-rank.

bad_query = "What monetary denomination do the forest inhabitants use for exchange?"
stage1_bad = dense_retrieve(bad_query, k=5)
good_query = "What is the Arborian currency?"
stage1_good = dense_retrieve(good_query, k=5)

print("Retrieval failure demo")
print("="*60)

# The Leaflet chunk is CORPUS[0] — find its rank under each query
currency_idx = 0  # CORPUS[0] is the Leaflet chunk

bad_ranks = [idx for idx, _ in stage1_bad]
good_ranks = [idx for idx, _ in stage1_good]

print(f"Vague query: {bad_query!r}")
print(f"  Rank of Leaflet chunk: "
      f"{bad_ranks.index(currency_idx)+1 if currency_idx in bad_ranks else 'not in top-5'}")
print(f"  Top-3 retrieved:")
for rank, (idx, score) in enumerate(stage1_bad[:3], 1):
    print(f"    [{rank}] score={score:.4f}  {CORPUS[idx][:65]}...")

print()
print(f"Clear query: {good_query!r}")
print(f"  Rank of Leaflet chunk: "
      f"{good_ranks.index(currency_idx)+1 if currency_idx in good_ranks else 'not in top-5'}")
print(f"  Top-3 retrieved:")
for rank, (idx, score) in enumerate(stage1_good[:3], 1):
    print(f"    [{rank}] score={score:.4f}  {CORPUS[idx][:65]}...")

In [ ]:
# --- Failure Mode 2: Generation failure ---
# Deliberately construct a context where the answer IS present
# but ask for something NOT there. A weak model may confabulate
# instead of saying "I don't know."

misleading_chunks = [
    CORPUS[4],   # Festival of Leaves — mentions spring, music
    CORPUS[12],  # Law about not felling trees
]
hallucination_query = "What is the Arborian national anthem?"

prompt_gen_fail = build_rag_prompt(hallucination_query, misleading_chunks)
answer_gen = chat(
    [{"role": "user", "content": prompt_gen_fail}],
    max_new_tokens=100,
    temperature=0.0,
)

print("Failure Mode 2 — Generation failure")
print("=" * 60)
print(f"Query: {hallucination_query!r}")
print(f"Context mentions: festival music, tree laws (nothing about an anthem)")
print(f"\nModel answer:")
print(answer_gen)
print()

# Detect hallucination vs. correct refusal
refused = "don't know" in answer_gen.lower() or "not mentioned" in answer_gen.lower() or "not in" in answer_gen.lower()
print(f"Correctly refused to answer: {refused}")
if not refused:
    print("  → GENERATION FAILURE: model hallucinated an answer not in context.")
    print("     Fix: use a stronger model, or add negative examples in the system prompt.")

In [ ]:
# --- Failure Mode 3: Context truncation ---
# Set a tiny token budget so the correct chunk gets cut off.

all_chunks = [CORPUS[i] for i in range(len(CORPUS))]
# Put the answer chunk last so truncation cuts it
shuffled = all_chunks[1:] + [CORPUS[0]]  # Leaflet chunk is last

tiny_budget = 80  # only fits ~2 chunks
context_str_tiny, n_included, tokens_used = build_context(shuffled, max_tokens=tiny_budget)

prompt_truncated = (
    "[CONTEXT]\n" + context_str_tiny + "\n[/CONTEXT]\n\n"
    "Answer the following question using only the context above. "
    "If the context does not contain the answer, say \"I don't know.\""
    "\n\nQuestion: What is the Arborian currency?"
)

answer_trunc = chat(
    [{"role": "user", "content": prompt_truncated}],
    max_new_tokens=80,
    temperature=0.0,
)

print("Failure Mode 3 — Context truncation")
print("=" * 60)
print(f"Token budget: {tiny_budget}")
print(f"Chunks included: {n_included} / {len(shuffled)} (answer chunk is last — truncated off)")
print(f"\nContext sent:")
print(context_str_tiny)
print(f"\nModel answer:")
print(answer_trunc)
print()
leaflet_in_context = "leaflet" in context_str_tiny.lower()
print(f"Leaflet chunk included in context: {leaflet_in_context}")
print("Fix: increase context budget, chunk the corpus more finely, or use a model with a larger context window.")

## 11. Try It Yourself

Three extensions that push the system from prototype to production-ready.

### Task A — Confidence Score

Add a `confidence_score` field to `RAGSystem.answer()` based on the top cross-encoder score. A calibration heuristic: score > 5 → high confidence, 1–5 → medium, < 1 → low.

*Hint*: the `top_reranker_score` key is already returned — just bucket it.

In [ ]:
# Task A — Add confidence score

def confidence_label(top_ce_score: float) -> str:
    """
    Map a cross-encoder score to a confidence label.
    CE scores from ms-marco-MiniLM are un-normalised logits — typically -10 to +10.
    """
    # YOUR CODE HERE
    # Hint: use if/elif/else on the score value
    raise NotImplementedError


# Once implemented, test it:
# for result in results:
#     label = confidence_label(result["top_reranker_score"])
#     print(f"{label:6s}  (score={result['top_reranker_score']})  Q: {result['question'][:50]}")

# Expected: Q5 (GDP) should be "low"; Q1-Q4 should be "high" or "medium".
print("Implement confidence_label() above, then un-comment the test block.")
print("Scores seen in the capstone run:",
      [r['top_reranker_score'] for r in results])

### Task B — Query Expansion

Generate 2 paraphrases of the question using the LLM, retrieve for all 3 queries, merge the candidate sets (deduplicating by index), and rerank the union.

This helps when the user's phrasing doesn't match the corpus vocabulary — a known weak point of dense retrieval.

In [ ]:
# Task B — Query expansion

def expand_query(question: str, n_paraphrases: int = 2) -> list[str]:
    """
    Use the LLM to generate `n_paraphrases` alternative phrasings of the question.
    Returns the original question plus the paraphrases.
    """
    # YOUR CODE HERE
    # Hint: ask the model for N alternative phrasings, parse them from the output.
    # Return [question] + [paraphrase1, paraphrase2, ...]
    raise NotImplementedError


def retrieve_with_expansion(question: str, k: int = 6) -> list[tuple[int, float]]:
    """
    Retrieve candidates for each query variant, then merge and deduplicate.
    Returns up to k unique candidates, ordered by best score across all queries.
    """
    # YOUR CODE HERE
    # 1. Call expand_query(question) to get query list
    # 2. For each query, call dense_retrieve(q, k=k)
    # 3. Merge: keep the best score seen for each corpus index
    # 4. Sort by score desc and return top-k
    raise NotImplementedError


# Once implemented, test:
# bad_query = "What monetary denomination do the forest inhabitants use for exchange?"
# expanded = expand_query(bad_query)
# print("Expanded queries:", expanded)
# candidates = retrieve_with_expansion(bad_query)
# reranked = rerank(bad_query, candidates, top_k=3)
# print("Top-3 after expansion + rerank:")
# for idx, score in reranked:
#     print(f"  [{idx}] {CORPUS[idx]}")
print("Implement expand_query() and retrieve_with_expansion() above.")
print("The goal: vague queries retrieve the correct chunk after expansion.")

### Task C — No-Answer Detector

Add a post-processing step that checks if the answer string contains "I don't know" (or close variants) and sets a boolean `no_answer` flag. Use this to route the question to a fallback (e.g., a web search or a human reviewer) rather than returning a low-confidence answer.

In [ ]:
# Task C — No-answer detector

NO_ANSWER_PHRASES = [
    "i don't know",
    "i do not know",
    "not mentioned",
    "not in the context",
    "cannot answer",
    "no information",
]

def is_no_answer(answer_text: str) -> bool:
    """
    Return True if the answer indicates the model couldn't find the information.
    Case-insensitive substring match against NO_ANSWER_PHRASES.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Once implemented, test on the capstone results:
# for result in results:
#     flag = is_no_answer(result["answer"])
#     status = "NO ANSWER" if flag else "answered"
#     print(f"[{status:9s}]  {result['question']}")
# Expected: Q5 (GDP) → NO ANSWER; Q1-Q4 → answered

print("Implement is_no_answer() above.")
print("Bonus: modify RAGSystem.answer() to include 'no_answer': bool in the return dict.")

## 12. Summary and What's Next

**What you built in this notebook:**

| Stage | Component | Key idea |
|---|---|---|
| Reranking | `CrossEncoder.predict(pairs)` | Joint (query, doc) attention — more precise than bi-encoder |
| Context budgeting | `build_context(chunks, max_tokens)` | Greedy packing with token estimation |
| Prompt construction | `build_rag_prompt(query, chunks)` | Delimiters + grounding instruction + I-don't-know fallback |
| Lost-in-the-middle | `order_chunks_for_context(chunks)` | Rank-1 first, rank-2 last |
| Citations | Numbered chunks + `parse_citations(answer)` | Traceability via inline `[N]` references |
| End-to-end system | `RAGSystem` class | Retrieve → rerank → budget → generate |
| Failure modes | Retrieval / generation / truncation | Know *which* stage broke to fix it correctly |

**The key architecture insight**: RAG is a two-tower system. The *retrieval tower* is optimised for recall (find anything that might help). The *generation tower* is optimised for faithfulness (say only what the context supports). They pull in opposite directions and the reranker is the bridge.

**What's next — Module E: Agents and Tool Use**

RAG gives a model access to a static knowledge base. The next step is giving it *tools* — functions it can call at inference time. An agent can search the web, run code, query a database, and then synthesise results. Module E builds that loop from scratch.

---
*Models used: `all-MiniLM-L6-v2` (bi-encoder), `cross-encoder/ms-marco-MiniLM-L-6-v2` (reranker), `HuggingFaceTB/SmolLM2-135M-Instruct` (generator). All components are modular — swap in a larger model at any stage.*